# Replication probe: group-ablation flip rate (Qwen2.5-VL-3B)

The cheapest piece of the replication gate (plan section 4, step 2). It does
**no patching**: it takes Lietzow's published promoting heads from
`configs/heads_lietzow_qwen3b.json`, zero-ablates them at the last token, and
measures how many prior-grounded predictions flip.

Their reported numbers for Qwen-VL 3B: **prior flip 84.9%**, visual flip 1.4%,
on **73** correctly-conflicting examples of 467.

If this lands near 84.9%, the pipeline -- prompts, tokenisation, head indexing,
ablation site, quantisation -- is right, and the full patching sweep is worth
its GPU hours. If it does not, fix that first: the sweep cannot diagnose it.

Precision is **4-bit** (D10), matching their default.

## 1. Setup

In [ ]:
REPO_URL = ""                                  # [FILL IN]
REPO_DIR = "/kaggle/working/research_paper"
import os, subprocess, sys
if REPO_URL and not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
!pip install -q -U "transformers>=4.49" accelerate datasets inflect bitsandbytes
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
from circuit_overlap.heads import load_head_set, format_head_label
hs = load_head_set(f"{REPO_DIR}/configs/heads_lietzow_qwen3b.json")
print(hs.name, hs.checkpoint)
print("promoting :", [format_head_label(h) for h in hs.promoting])
print("suppressing:", [format_head_label(h) for h in hs.suppressing])
print("layer range:", hs.layer_range(), " reported prior flip: 84.9%")

In [ ]:
from circuit_overlap.t_screen import ScreenConfig, VISUAL_PROMPT, PRIOR_PROMPT, load_model
config = ScreenConfig()
model, processor = load_model(config)        # 4-bit, nf4, double quant (D10)

# [VERIFY] the head geometry before ablating anything: a mismatch here is the
# single most likely cause of a failed gate.
text_config = model.model.language_model.config
n_layers = text_config.num_hidden_layers
n_heads = text_config.num_attention_heads
head_dim = getattr(text_config, "head_dim", text_config.hidden_size // n_heads)
print(f"{n_layers} layers x {n_heads} heads, head_dim={head_dim}")
load_head_set(f"{REPO_DIR}/configs/heads_lietzow_qwen3b.json",
              n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if 1-based

## 2. The conflicting subset

Per their README, the conflicting subset is the examples whose **top-1
prediction differs between the two prompts**. Build the manifest with
`notebooks/t_screen.ipynb` cell 2 first (it reproduces their row filters), then
reuse it here -- the P condition needs the **counterfact** image, so keep the
dataset handy rather than only the saved PNGs.

In [ ]:
# [FILL IN] load the dataset and the manifest as in t_screen.ipynb cell 2,
# keeping both images per row. The P contrast is:
#   Visual prompt + counterfactual image   vs   Prior prompt + counterfactual image
# with the answer-format suffix already included in both constants.
print(repr(VISUAL_PROMPT)); print(repr(PRIOR_PROMPT))

## 3. Ablation

Lietzow zero the **o_proj input slice** `[head*head_dim : (head+1)*head_dim]`
at the **last token** (`src/vdpo/interventions.py::head_knockout`, MIT). Their
file is short and directly reusable with attribution (D65) -- prefer that to a
reimplementation, so the intervention site cannot drift.

Flip rate = fraction of prior-grounded predictions whose top-1 answer changes
once the promoting heads are ablated; `circuit_overlap.metrics.flip_rate`
computes it, with the conflicting subset as the mask.

In [ ]:
from circuit_overlap.metrics import flip_rate
# [FILL IN] baseline and ablated top-1 answers over the conflicting subset,
# then:
#   prior  = flip_rate(prior_before,  prior_after)    # expect ~0.849
#   visual = flip_rate(visual_before, visual_after)   # expect ~0.014

## 4. Record it

Write the result through `provenance.result_envelope` (model, checkpoint, n,
seed, prompt, commit) and add a row to `docs/RESULTS.md`. Note the compute
dtype: a T4 has no bf16, so 4-bit compute falls back to fp16 where Lietzow's
runs were probably bf16.

| Outcome | Reading |
|---|---|
| ~84.9% prior, ~1.4% visual | Pipeline verified; proceed to the patching sweep |
| Prior flip far lower | Check head indexing, the ablation site, the prompt suffix, and the conflicting-subset definition before anything else |
| Visual flip high | The ablation is hitting more than the intended heads |